# 🧹 Data Cleaning

> ### 📌 Project Approach
<br>
📂 **Load the Datasets** → `2016` + `2018` <br><br> 
⬇️  
🔗 **Append Both Datasets** → `all_data`  <br><br>
⬇️  
🧹 **Clean the Data**  <br><br>
⬇️  
💾 **Save the Cleaned Data** → `.csv`

In [1]:
# approach 
"""
📌 #[ problems while merge preperations ]

(1) column numbers 
- delete extra columns from (2018)

(2) columns names 
- remove extra space at the end of column from (2016) 


(3) shifted column values problem in (2016): 
    merge the splited name values and append it to name column 
    reassign other column values in the correct order 
    drop the extra 4 columns at the end 

(4) unsuitable column data types 
    convert to the suitable 
0   ID                int64         
1   name             object        
2   category         object        
3   main_category    object        
4   currency         object        
5   deadline         datetime64[ns]
6   goal             float64       
7   launched         datetime64[ns]
8   pledged          float64       
9   state            object        
10  backers          int64         
11  country          object        
12  usd pledged      float64 


📌 [ problems while cleaning all_data ]

(1) handle nulls  
- name : remove nulls
- country : replace N,O" with undfiend
- usd pledged : 
    USD Country: 
        replace with the pledged value column 
    other Contries: create an algorithm to convert from its currency to usd currency
        extract equation to convert into usd currency from data based on date and currency
    #*the currency conversion algorithm graph is shown at the end of the notebook
"""

'\n📌 #[ problems while merge preperations ]\n\n(1) column numbers \n- delete extra columns from (2018)\n\n(2) columns names \n- remove extra space at the end of column from (2016) \n\n\n(3) shifted column values problem in (2016): \n    merge the splited name values and append it to name column \n    reassign other column values in the correct order \n    drop the extra 4 columns at the end \n\n(4) unsuitable column data types \n    convert to the suitable \n0   ID                int64         \n1   name             object        \n2   category         object        \n3   main_category    object        \n4   currency         object        \n5   deadline         datetime64[ns]\n6   goal             float64       \n7   launched         datetime64[ns]\n8   pledged          float64       \n9   state            object        \n10  backers          int64         \n11  country          object        \n12  usd pledged      float64 \n\n\n📌 [ problems while cleaning all_data ]\n\n(1) handle null

In [2]:
import pandas as pd 
import numpy as np 

In [3]:

df_2016 = pd.read_csv(
    r"E:\Documents\Data_Science\Schoralships\ITI-BI-2026\PowerBI\Lab2\raw_data\ks-projects-201612.csv",
    encoding="latin1"
)

df_2018 = pd.read_csv(r"E:\Documents\Data_Science\Schoralships\ITI-BI-2026\PowerBI\Lab2\raw_data\ks-projects-201801.csv", 
    encoding="latin1")

C:\Users\ASUD Vivo\AppData\Local\Temp\ipykernel_2912\1988156634.py:1: DtypeWarning: Columns (13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2016 = pd.read_csv(


## df_2016 append preperation

### columns names

In [4]:
# remove the extra space in columns names
df_2016.columns = df_2016.columns.str.strip()

### Handling shifted columns problem 

```
🔄 Overall Process
Incorrectly Shifted Row
        │
        ▼
Identify Extra Name Parts
        │
        ▼
Reconstruct Project Name
        │
        ▼
Shift Remaining Values Left
        │
        ▼
Remove Extra Columns
        │
        ▼
Corrected Row
```

In [5]:
df_2016.loc[:, ['Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']]
errors_filt = df_2016['Unnamed: 13'].notna()
errors_filt.sum()



np.int64(625)

In [6]:
idx_errors  = errors_filt[errors_filt].index.to_numpy()
copy_df_2016 = df_2016.copy()

num_shifted_columns = (
    copy_df_2016[errors_filt]
    .iloc[:, -4:]
    .notna()
    .sum(axis=1)
)

for idx, n in zip(idx_errors, num_shifted_columns):

    # Get the values that belong to the name
    splited_name_values = copy_df_2016.iloc[idx, 2:2+n].tolist()

    # Append them to the name
    copy_df_2016.iloc[idx, 1] += "," + ",".join(map(str, splited_name_values))

    # Get the remaining values
    remain_row_values = copy_df_2016.iloc[idx, 2+n:].to_numpy()

    # Shift them left
    copy_df_2016.iloc[idx, 2:2+len(remain_row_values)] = remain_row_values

    # Optional: clear the leftover cells at the end
    copy_df_2016.iloc[idx, 2+len(remain_row_values):] = np.nan

copy_df_2016.drop(columns = copy_df_2016.columns[-4:], inplace = True)
copy_df_2016[errors_filt]

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged
1454,1008705746,"Zephyra´s new full length, 'As The World Colla...",Metal,Music,SEK,2016-02-02 00:56:46,15000,2016-01-03 00:56:46,4262,failed,14,SE,504.94765278
1563,1009317190,"French Cuisine, A Traditional Experience",Cookbooks,Food,USD,2014-09-08 00:46:23,13730,2014-08-09 03:16:02,3984,failed,46,US,3984
1794,1010871699,"The Beginners Guide to being Unsuicidal, the o...",Theater,Theater,USD,2011-12-31 23:25:46,5000,2011-11-21 23:25:46,525,failed,10,US,525
1931,1011687764,"Best OnLine Classifieds, Ever / No More Spam",Web,Technology,USD,2014-09-20 19:56:10,6300,2014-08-21 19:56:10,0,failed,0,US,0
2420,101453314,"Social Media Ruined My Life, A Short Film from...",Shorts,Film & Video,USD,2013-03-14 20:11:57,3000,2013-02-25 21:11:57,3035,successful,42,US,3035
...,...,...,...,...,...,...,...,...,...,...,...,...,...
321945,989007729,"THROUGH MY EYES, MY LIFE IN THE MISSISSIPPI DELTA",Narrative Film,Film & Video,USD,2012-04-05 02:34:49,3500,2012-03-06 02:34:49,10,failed,1,US,10
322162,990511774,"Daniel Hresko's new CD is (almost) ready, so g...",Indie Rock,Music,USD,2011-09-06 05:59:00,400,2011-08-06 15:36:39,61,failed,4,US,61
322204,990746749,"Feet on the Ground, Head in the Clouds",Film & Video,Film & Video,USD,2013-04-17 16:00:31,35000,2013-03-18 15:00:31,179,failed,10,US,179
323138,996542939,"'WANDER' - Apocalyptic Short Film, Post Produc...",Shorts,Film & Video,GBP,2015-04-26 17:12:59,1500,2015-02-25 17:12:59,2501,successful,50,GB,3861.84447014


### compined problem: the shifted columns and usd pludged nulls

In [7]:
#check other errros 
check = ~ pd.to_numeric(copy_df_2016['pledged'], errors='coerce').notna()
copy_df_2016[check]

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged
64484,1383759393,CD: Heartsong Harmonics,Marc & Mary sing together at last!,Music,Music,USD,2015-11-03 00:59:00,7085,2015-09-14 23:55:04,7470,undefined,0,"N,""0"
85619,1508366697,Everyday Beautiful,a day in the life of a champion,Film & Video,Film & Video,USD,2015-03-27 01:00:00,2000,2015-02-25 06:52:16,0,undefined,0,"N,""0"
98614,1585608231,Hana Zara's new album,The North (2015),Music,Music,USD,2015-03-03 02:04:57,3000,2015-01-17 02:04:57,3280,undefined,0,"N,""0"
123506,1733391784,Celebrating Joni Mitchell,Songs by & about her + Stories,Music,Music,CAD,2015-10-11 00:47:47,5000,2015-09-11 00:47:47,6323,undefined,0,"N,""0"
159427,194816108,Legal Highs,The Sobering Truth,Film & Video,Film & Video,GBP,2014-11-05 03:00:57,5000,2014-10-16 03:00:57,0,undefined,0,"N,""0"
163067,1969863991,Puppy Training Steps,Right and Wrong,Film & Video,Film & Video,USD,2015-04-24 22:07:03,5000,2015-03-25 21:07:03,0,undefined,0,"N,""0"
177636,2057841246,Adventure to Peru's Sacred Valley,Recording Soundtracks,Music,Music,USD,2015-03-21 21:13:23,2200,2015-03-02 22:14:34,2503,undefined,0,"N,""0"


In [8]:

idx_errors = copy_df_2016[check].index

for idx in idx_errors:

    # Join the 3rd and 4th columns into the 3rd column
    copy_df_2016.iloc[idx, 2] = (
        str(copy_df_2016.iloc[idx, 2]) + " " +
        str(copy_df_2016.iloc[idx, 3])
    )

    # Shift remaining columns one position to the left
    copy_df_2016.iloc[idx, 3:-1] = (
        copy_df_2016.iloc[idx, 4:].to_numpy()
    )

In [9]:
copy_df_2016.loc[check.index]

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged
0,1000002330,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09 11:36:00,1000,2015-08-11 12:12:28,0,failed,0,GB,0
1,1000004038,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26 00:20:50,45000,2013-01-12 00:20:50,220,failed,3,US,220
2,1000007540,ToshiCapital Rekordz Needs Help to Complete Album,Music,Music,USD,2012-04-16 04:24:11,5000,2012-03-17 03:24:11,1,failed,1,US,1
3,1000011046,Community Film Project: The Art of Neighborhoo...,Film & Video,Film & Video,USD,2015-08-29 01:00:00,19500,2015-07-04 08:35:03,1283,canceled,14,US,1283
4,1000014025,Monarch Espresso Bar,Restaurants,Food,USD,2016-04-01 13:38:27,50000,2016-02-26 13:38:27,52375,successful,224,US,52375
...,...,...,...,...,...,...,...,...,...,...,...,...,...
323745,999976400,ChknTruk Nationwide Charity Drive 2014 (Canceled),Documentary,Film & Video,USD,2014-10-17 02:35:30,50000,2014-09-17 02:35:30,25,canceled,1,US,25
323746,999977640,The Tribe,Narrative Film,Film & Video,USD,2011-07-19 03:35:14,1500,2011-06-22 03:35:14,155,failed,5,US,155
323747,999986353,Walls of Remedy- New lesbian Romantic Comedy f...,Narrative Film,Film & Video,USD,2010-08-16 05:59:00,15000,2010-07-01 19:40:30,20,failed,1,US,20
323748,999987933,BioDefense Education Kit,Technology,Technology,USD,2016-02-13 02:00:00,15000,2016-01-13 18:13:53,200,failed,6,US,200


In [10]:

df_2016 = copy_df_2016.copy()
df_2016.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323750 entries, 0 to 323749
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   ID             323750 non-null  int64 
 1   name           323746 non-null  object
 2   category       323750 non-null  object
 3   main_category  323750 non-null  object
 4   currency       323750 non-null  object
 5   deadline       323750 non-null  object
 6   goal           323750 non-null  object
 7   launched       323750 non-null  object
 8   pledged        323750 non-null  object
 9   state          323750 non-null  object
 10  backers        323750 non-null  object
 11  country        323750 non-null  object
 12  usd pledged    319960 non-null  object
dtypes: int64(1), object(12)
memory usage: 32.1+ MB


### add tag

In [11]:
df_2016["year"] = "2016"

### data types correction

In [12]:
# Expected data types
columns_types = {
    "goal": "numeric",
    "pledged": "numeric",
    "backers": "numeric",
    "usd pledged": "numeric",
    "deadline": "datetime",
    "launched": "datetime"
}

# Dictionary to store error masks
error_masks = {}

for col, dtype in columns_types.items():
    
    if dtype == "numeric":
        converted = pd.to_numeric(df_2016[col], errors="coerce")
        
        # True = conversion failed (but original value was not null)
        error_masks[f"{col}_error_mask"] = (
            df_2016[col].notna() & converted.isna()
        )
        
        # Apply conversion
        df_2016[col] = converted

    
    elif dtype == "datetime":
        converted = pd.to_datetime(df_2016[col], errors="coerce")
        
        # True = conversion failed
        error_masks[f"{col}_error_mask"] = (
            df_2016[col].notna() & converted.isna()
        )
        
        # Apply conversion
        df_2016[col] = converted

In [13]:
object_cols = df_2016.select_dtypes(include="object").columns
df_2016[object_cols] = df_2016[object_cols].astype("string")

In [14]:
df_2016.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 323750 entries, 0 to 323749
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             323750 non-null  int64         
 1   name           323746 non-null  string        
 2   category       323750 non-null  string        
 3   main_category  323750 non-null  string        
 4   currency       323750 non-null  string        
 5   deadline       323750 non-null  datetime64[ns]
 6   goal           323750 non-null  float64       
 7   launched       323750 non-null  datetime64[ns]
 8   pledged        323750 non-null  float64       
 9   state          323750 non-null  string        
 10  backers        323750 non-null  float64       
 11  country        323750 non-null  string        
 12  usd pledged    319953 non-null  float64       
 13  year           323750 non-null  string        
dtypes: datetime64[ns](2), float64(4), int64(1), string(7

## df_2018 append preperation


In [15]:
df_2018.columns

Index(['ID', 'name', 'category', 'main_category', 'currency', 'deadline',
       'goal', 'launched', 'pledged', 'state', 'backers', 'country',
       'usd pledged', 'usd_pledged_real', 'usd_goal_real'],
      dtype='object')

###  number of columns append problem hanling 

In [16]:
df_2018.drop(columns= ['usd_pledged_real', 'usd_goal_real'], inplace = True)

### add tag 

In [17]:
df_2018["year"] = "2018"

In [18]:
df_2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378661 entries, 0 to 378660
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   ID             378661 non-null  int64  
 1   name           378657 non-null  object 
 2   category       378661 non-null  object 
 3   main_category  378661 non-null  object 
 4   currency       378661 non-null  object 
 5   deadline       378661 non-null  object 
 6   goal           378661 non-null  float64
 7   launched       378661 non-null  object 
 8   pledged        378661 non-null  float64
 9   state          378661 non-null  object 
 10  backers        378661 non-null  int64  
 11  country        378661 non-null  object 
 12  usd pledged    374864 non-null  float64
 13  year           378661 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 40.4+ MB


### column data type correction

In [19]:
object_cols = df_2018.select_dtypes(include="object").columns

df_2018[object_cols] = df_2018[object_cols].astype("string")

In [20]:


# Expected data types
columns_types = {
    "goal": "numeric",
    "pledged": "numeric",
    "backers": "numeric",
    "usd pledged": "numeric",
    "deadline": "datetime",
    "launched": "datetime"
}

# Dictionary to store error masks
error_masks = {}

for col, dtype in columns_types.items():
    
    if dtype == "numeric":
        converted = pd.to_numeric(df_2018[col], errors="coerce")
        
        # True = conversion failed (but original value was not null)
        error_masks[f"{col}_error_mask"] = (
            df_2018[col].notna() & converted.isna()
        )
        
        # Apply conversion
        df_2018[col] = converted

    
    elif dtype == "datetime":
        converted = pd.to_datetime(df_2018[col], errors="coerce")
        
        # True = conversion failed
        error_masks[f"{col}_error_mask"] = (
            df_2018[col].notna() & converted.isna()
        )
        
        # Apply conversion
        df_2018[col] = converted

In [21]:
for name, value in error_masks.items():
    print(name)
    print("Number of errors", value.sum())

goal_error_mask
Number of errors 0
pledged_error_mask
Number of errors 0
backers_error_mask
Number of errors 0
usd pledged_error_mask
Number of errors 0
deadline_error_mask
Number of errors 0
launched_error_mask
Number of errors 0


In [22]:
df_2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378661 entries, 0 to 378660
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             378661 non-null  int64         
 1   name           378657 non-null  string        
 2   category       378661 non-null  string        
 3   main_category  378661 non-null  string        
 4   currency       378661 non-null  string        
 5   deadline       378661 non-null  datetime64[ns]
 6   goal           378661 non-null  float64       
 7   launched       378661 non-null  datetime64[ns]
 8   pledged        378661 non-null  float64       
 9   state          378661 non-null  string        
 10  backers        378661 non-null  int64         
 11  country        378661 non-null  string        
 12  usd pledged    374864 non-null  float64       
 13  year           378661 non-null  string        
dtypes: datetime64[ns](2), float64(3), int64(2), string(7

## merge two datasets

In [23]:
all_data = pd.concat([df_2016, df_2018], ignore_index=True)
all_data

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,year
0,1000002330,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09 11:36:00,1000.0,2015-08-11 12:12:28,0.0,failed,0.0,GB,0.0,2016
1,1000004038,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26 00:20:50,45000.0,2013-01-12 00:20:50,220.0,failed,3.0,US,220.0,2016
2,1000007540,ToshiCapital Rekordz Needs Help to Complete Album,Music,Music,USD,2012-04-16 04:24:11,5000.0,2012-03-17 03:24:11,1.0,failed,1.0,US,1.0,2016
3,1000011046,Community Film Project: The Art of Neighborhoo...,Film & Video,Film & Video,USD,2015-08-29 01:00:00,19500.0,2015-07-04 08:35:03,1283.0,canceled,14.0,US,1283.0,2016
4,1000014025,Monarch Espresso Bar,Restaurants,Food,USD,2016-04-01 13:38:27,50000.0,2016-02-26 13:38:27,52375.0,successful,224.0,US,52375.0,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
702406,999976400,ChknTruk Nationwide Charity Drive 2014 (Canceled),Documentary,Film & Video,USD,2014-10-17 00:00:00,50000.0,2014-09-17 02:35:30,25.0,canceled,1.0,US,25.0,2018
702407,999977640,The Tribe,Narrative Film,Film & Video,USD,2011-07-19 00:00:00,1500.0,2011-06-22 03:35:14,155.0,failed,5.0,US,155.0,2018
702408,999986353,Walls of Remedy- New lesbian Romantic Comedy f...,Narrative Film,Film & Video,USD,2010-08-16 00:00:00,15000.0,2010-07-01 19:40:30,20.0,failed,1.0,US,20.0,2018
702409,999987933,BioDefense Education Kit,Technology,Technology,USD,2016-02-13 00:00:00,15000.0,2016-01-13 18:13:53,200.0,failed,6.0,US,200.0,2018


In [24]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 702411 entries, 0 to 702410
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             702411 non-null  int64         
 1   name           702403 non-null  string        
 2   category       702411 non-null  string        
 3   main_category  702411 non-null  string        
 4   currency       702411 non-null  string        
 5   deadline       702411 non-null  datetime64[ns]
 6   goal           702411 non-null  float64       
 7   launched       702411 non-null  datetime64[ns]
 8   pledged        702411 non-null  float64       
 9   state          702411 non-null  string        
 10  backers        702411 non-null  float64       
 11  country        702411 non-null  string        
 12  usd pledged    694817 non-null  float64       
 13  year           702411 non-null  string        
dtypes: datetime64[ns](2), float64(4), int64(1), string(7

### Checking nulls and empry str

In [25]:
# Get all object columns
object_cols = all_data.select_dtypes(include="object").columns

# Strip whitespace from all object columns
all_data[object_cols] = all_data[object_cols].apply(lambda col: col.str.strip())

# Replace empty strings with NaN
all_data[object_cols] = all_data[object_cols].replace("", np.nan)

# Count missing values
all_data.isna().sum()

ID                  0
name                8
category            0
main_category       0
currency            0
deadline            0
goal                0
launched            0
pledged             0
state               0
backers             0
country             0
usd pledged      7594
year                0
dtype: int64

#### nulls  
- name = 8,   | remove

- usd pledged = 3797  | investegate more


In [26]:
all_data.dropna(subset=["name"], inplace=True)

In [27]:
use_null_filt = all_data["usd pledged"].isna()
all_data[use_null_filt].loc[:, ["currency", "pledged", "usd pledged", "country", "state"]]

,currency,pledged,usd pledged,country,state
150,USD,555.00,NaN,"N,""0",undefined
287,AUD,4767.00,NaN,"N,""0",undefined
549,USD,3576.00,NaN,"N,""0",undefined
561,USD,7007.80,NaN,"N,""0",undefined
650,USD,3660.38,NaN,"N,""0",undefined
...,...,...,...,...,...
701983,USD,10.00,NaN,"N,0""",undefined
702053,CAD,3102.00,NaN,"N,0""",undefined
702184,USD,235.00,NaN,"N,0""",undefined
702335,GBP,2125.00,NaN,"N,0""",undefined


### problem observation [Country has N,O" value]
- check if country could be correctly generated by currency

In [28]:
currency_country_count = (
    all_data.groupby("currency")["country"]
    .nunique()
)

currency_country_count[currency_country_count > 1]

currency
AUD     3
CAD     3
CHF     3
DKK     3
EUR    11
GBP     3
NOK     3
NZD     3
SEK     3
USD     3
Name: country, dtype: int64

#### invalid to generate country name based on currency ❌ 
- cause same currency used in multiple countries <br>

#### taken action -> replace N,O" with undefined <br><br>

In [29]:
all_data["country"] = all_data["country"].replace('N,0"', "Undefined")

#### usd_pledged nulls handling 

```
START
  │
  ▼
All Data
  │
  ├── If pledged = 0
  │       └── Set usd pledged = 0
  │
  ├── If country = US
  │       └── Set usd pledged = pledged
  ▼
Split Data into Two Parts
  │
  ├──────────────────────────┐
  ▼                          ▼
Invalid Rows               Valid Rows
usd pledged = NaN          usd pledged ≠ NaN
pledged ≠ NaN              pledged ≠ NaN
pledged ≠ 0                pledged ≠ 0
  │                          │
  │                          ▼
  │                    Group Valid Rows
  │                    by:
  │                    Currency + Launch Date
  │                          │
  └──────────────┬───────────┘
                 │
                 ▼
       For Each Invalid Row
                 │
                 ▼
      Search Valid Data for:
      • Same Currency
      • Within ±3 Days
                 │
                 ▼
          Any Match Found?
             │        │
            No       Yes
             │        │
            ▼        ▼
           NaN    Exact Date?
                    │      │
                   Yes     No
                    │      │
                    ▼      ▼
               Use Exact   Use Closest
                 Row         Row
                    │      │
                    └──┬───┘
                       │
                       ▼
            Calculate Multiplier
            usd pledged / pledged
                       │
                       ▼
       Current pledged × Multiplier
                       │
                       ▼
          Store in usd pledged
                       │
                       ▼
                      END
```

In [30]:
zero_pledged_filt = (
    all_data["pledged"].notna() &
    (all_data["pledged"] == 0)
)

all_data.loc[
    zero_pledged_filt,
    "usd pledged"
] = 0

# stor US usd pledged as the original pledged
all_data.loc[
    all_data["pledged"].notna() & (all_data["country"] == "US"),
    "usd pledged"
] = all_data["pledged"]


In [31]:


# ==========================================
# 1. Invalid data
# usd pledged = null
# pledged exists
# pledged != 0
# ==========================================

invalid_filt = (
    all_data["usd pledged"].isna() &
    all_data["pledged"].notna() &
    (all_data["pledged"] != 0)
)

invalid_data = all_data.loc[invalid_filt].copy()


# ==========================================
# 2. Valid data
# usd pledged exists
# pledged exists
# pledged != 0
# ==========================================

valid_filt = (
    all_data["usd pledged"].notna() &
    all_data["pledged"].notna() &
    (all_data["pledged"] != 0)
)

valid_data = all_data.loc[valid_filt]


# ==========================================
# 3. Group valid data
# by currency and launched date
# ==========================================

valid_grouped = valid_data.groupby(
    ["currency", "launched"]
)

In [32]:

# ==========================================
# 4. Function to calculate missing usd pledged
# ==========================================

def calculate_usd_pledged(row):

    current_currency = row["currency"]
    current_date = row["launched"]
    current_pledged = row["pledged"]

    
    # ==========================================
    # 1. Define ±3 days search period
    # ==========================================

    start_date = current_date - pd.Timedelta(days=3)
    end_date = current_date + pd.Timedelta(days=3)


    # ==========================================
    # 2. Find same currency within ±3 days
    # ==========================================

    similar_rows = valid_data[
        (valid_data["currency"] == current_currency) &
        (valid_data["launched"] >= start_date) &
        (valid_data["launched"] <= end_date)
    ]


    # No rows found
    if similar_rows.empty:
        return np.nan


    # ==========================================
    # 3. Look for exact date match
    # ==========================================

    exact_match = similar_rows[
        similar_rows["launched"] == current_date
    ]


    # ==========================================
    # 4. If exact match exists → use first row
    # ==========================================

    if not exact_match.empty:

        matched_row = exact_match.iloc[0]


    # ==========================================
    # 5. Otherwise → find closest date
    # ==========================================

    else:

        date_difference = (
            similar_rows["launched"] - current_date
        ).abs()

        closest_idx = date_difference.idxmin()

        matched_row = similar_rows.loc[closest_idx]


    # ==========================================
    # 6. Calculate multiplier
    # ==========================================

    multiplier = (
        matched_row["usd pledged"] /
        matched_row["pledged"]
    )


    # ==========================================
    # 7. Calculate missing USD pledged
    # ==========================================

    return current_pledged * multiplier

In [33]:
invalid_data["usd pledged"] = invalid_data.apply(
    calculate_usd_pledged,
    axis=1
)

# ==========================================
# 6. Store results back in all_data
# ==========================================

all_data.loc[
    invalid_data.index,
    "usd pledged"
] = invalid_data["usd pledged"]

In [34]:
all_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 702403 entries, 0 to 702410
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             702403 non-null  int64         
 1   name           702403 non-null  string        
 2   category       702403 non-null  string        
 3   main_category  702403 non-null  string        
 4   currency       702403 non-null  string        
 5   deadline       702403 non-null  datetime64[ns]
 6   goal           702403 non-null  float64       
 7   launched       702403 non-null  datetime64[ns]
 8   pledged        702403 non-null  float64       
 9   state          702403 non-null  string        
 10  backers        702403 non-null  float64       
 11  country        702403 non-null  string        
 12  usd pledged    702403 non-null  float64       
 13  year           702403 non-null  string        
dtypes: datetime64[ns](2), float64(4), int64(1), string(7)
mem

In [ ]:
all_data.to_csv(r"E:\Documents\Data_Science\Schoralships\ITI-BI-2026\PowerBI\Lab2\Cleaned\Cleaned.csv", index= False)